# 3D Deviatoric Maxwell Viscoelasticity with SINDy

This notebook reproduces the 3D viscoelastic benchmark using the deviatoric formulation of the isotropic Maxwell model.

The workflow is:

1. Define tensor utility functions.
2. Generate synthetic 3D deviatoric Maxwell datasets.
3. Train component-wise SINDy models.
4. Validate on unseen combined and biaxial loading paths.
5. Map the results to Figures 4 and 5.

The key point is that the Maxwell law is applied to the deviatoric stress tensor, not independently to the full six-component Voigt stress vector.

In [1]:
import os
os.chdir('./ViscoElasticity_tensor_3D/archive')
print(os.getcwd())
print(os.listdir())

c:\Users\qq528\OneDrive - KTH\WorkingFolder\Courses\FSM3001_data_driven\sindy_project\Emanuele\Ema\ViscoElasticity_tensor_3D\archive
['3d_deviatoric_data.png', 'check_positive.py', 'data_train_shear_dev.npz', 'data_train_uniaxial_dev.npz', 'data_val_biaxial_dev.npz', 'data_val_combined_dev.npz', 'deviatoric_sindy_results.png', 'deviatoric_utils.py', 'diagnose_issue.py', 'Figure_3.png', 'generate_deviatoric_data.py', 'inputOutput_training.png', 'Input_output_validation.png', 'Readme', 'simple_visco_sindy.py', 'simple_visco_sindy_deviatoric.py', 'Validation_training.png', 'von_mises_utils.py']


# `deviatoric_utils.py` — Tensor utilities for the 3D deviatoric Maxwell model

**Related to reviewer comment 5.**

This file defines the deviatoric stress and deviatoric strain-rate operations used by the corrected 3D viscoelastic model.

The stress decomposition is

$$
\boldsymbol{\sigma}
=
p\boldsymbol{I}
+
\boldsymbol{S},
\qquad
p=\frac{1}{3}\operatorname{tr}(\boldsymbol{\sigma}),
$$

so that

$$
\boldsymbol{S}
=
\boldsymbol{\sigma}
-
\frac{1}{3}\operatorname{tr}(\boldsymbol{\sigma})\boldsymbol{I}.
$$

The strain-rate decomposition is

$$
\dot{\boldsymbol{\varepsilon}}^{dev}
=
\dot{\boldsymbol{\varepsilon}}
-
\frac{1}{3}\operatorname{tr}(\dot{\boldsymbol{\varepsilon}})\boldsymbol{I}.
$$

These functions are required because the corrected 3D Maxwell model is formulated in deviatoric space.

# `generate_deviatoric_data.py` — Generate corrected 3D viscoelastic datasets

**Solution to reviewer comment 5.**

This file generates the datasets used for Figures 4 and 5. The model is the deviatoric formulation of the 3D isotropic Maxwell model:

$$
\dot{\boldsymbol{S}}
=
2G\dot{\boldsymbol{\varepsilon}}^{dev}
-
\frac{2G}{\eta}\boldsymbol{S}.
$$

Component-wise,

$$
\dot{S}_{ij}
=
2G\dot{\varepsilon}_{ij}^{dev}
-
\frac{2G}{\eta}S_{ij}.
$$

The parameters are

$$
G=1000,\qquad \eta=500.
$$

Therefore, the expected SINDy coefficients are

$$
c_1=-\frac{2G}{\eta}=-4,
\qquad
c_2=2G=2000.
$$

This script creates:

```text
data_train_uniaxial_dev.npz
data_train_shear_dev.npz
data_val_combined_dev.npz
data_val_biaxial_dev.npz

In [ ]:
!python generate_deviatoric_data.py

### `simple_visco_sindy.py` — 1D Maxwell baseline

This file generates and identifies the 1D Maxwell model:

$$
\dot{\sigma}
=
E\dot{\varepsilon}
-
\frac{E}{\eta}\sigma.
$$

SINDy identifies

$$
\dot{\sigma}
=
c_0+c_1\sigma+c_2\dot{\varepsilon}.
$$

The expected coefficients are

$$
c_0=0,
\qquad
c_1=-\frac{E}{\eta},
\qquad
c_2=E.
$$

This script corresponds to the 1D viscoelastic benchmark, not Figures 4 and 5.

In [ ]:
!python simple_visco_sindy.py

<figure>
  <img src="./ViscoElasticity_tensor_3D/archive/Figure_3.png" alt="Figure 3" width="600">
  <figcaption>Figure 3 : Stress–strain response for training and validation datasets.</figcaption>
</figure>


---


### `simple_visco_sindy_deviatoric.py` — Component-wise 3D SINDy fit



This file performs the missing component-wise 3D SINDy fitting and produces the validation results for Figures 4 and 5.

The SINDy model is

$$
\dot{S}_{ij}
=
c_0+c_1S_{ij}+c_2\dot{\varepsilon}_{ij}^{dev}.
$$

From the deviatoric Maxwell model,

$$
\dot{S}_{ij}
=
2G\dot{\varepsilon}_{ij}^{dev}
-
\frac{2G}{\eta}S_{ij},
$$

so the expected coefficients are

$$
c_0=0,
\qquad
c_1=-\frac{2G}{\eta},
\qquad
c_2=2G.
$$

The script trains:

$$
\dot{S}_{xx}
=
c_0+c_1S_{xx}+c_2\dot{\varepsilon}_{xx}^{dev}
$$

from the uniaxial dataset, and

$$
\dot{S}_{xy}
=
c_0+c_1S_{xy}+c_2\dot{\gamma}_{xy}
$$

from the shear dataset.

It then validates on:


combined uniaxial + shear  → Figure 4
biaxial loading            → Figure 5

In [ ]:
!python simple_visco_sindy_deviatoric.py

'[Stress--strain' is not recognized as an internal or external command,
operable program or batch file.


<figure>
  <img src="./ViscoElasticity_tensor_3D/archive/Figure_4.png" alt="Figure 3" width="600">
  <figcaption>Figure 4 : Validation for the combined case of tensile and shear loads.</figcaption>
</figure>

<figure>
  <img src="./ViscoElasticity_tensor_3D/archive/Figure_5.png" alt="Figure 3" width="600">
  <figcaption>Figure 5 : Validation for the combined case of biaxial load.</figcaption>
</figure>